In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from collections import Counter
 
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
 
os.environ["OMP_NUM_THREADS"] = "3"

In [ ]:
with zipfile.ZipFile("../data/archive (1).zip", 'r') as zip_ref:
    zip_ref.extractall("../data")
 
results   = pd.read_csv("../data/results.csv")
qualifying = pd.read_csv("../data/qualifying.csv")
races      = pd.read_csv("../data/races.csv")
drivers    = pd.read_csv("../data/drivers.csv")

In [ ]:
df = pd.merge(
    results,
    qualifying,
    on=["driverId", "raceId"],
    how="inner",
    suffixes=("_race", "_qual")
)
 
df = pd.merge(df, races[["raceId", "year"]], on="raceId", how="left")
 
df["position_change"] = df["position_qual"] - df["positionOrder"]

In [ ]:
race_laps = results.groupby("raceId")["laps"].max().reset_index()
race_laps = race_laps.rename(columns={"laps": "total_laps"})
df = pd.merge(df, race_laps, on="raceId", how="left")
df["completion_pct"] = df["laps"] / df["total_laps"].clip(lower=1)

In [ ]:
df = df[df["positionOrder"] > 0]
df = df[df["position_qual"] > 0]

In [ ]:
def compute_behavioral_features(df_season, groupby_col="driverId", min_races=5):
    g = df_season.groupby(groupby_col)

    feats = pd.DataFrame()

    feats["avg_gain"] = g["position_change"].mean()
    feats["gain_pct"] = g["position_change"].apply(lambda x: (x > 0).mean())
    feats["volatility"] = g["position_change"].std()
    feats["p10_gain"] = g["position_change"].quantile(0.10)
    feats["p90_gain"] = g["position_change"].quantile(0.90)

    
    feats["avg_qual"] = g["position_qual"].mean()
    feats["gain_per_qual_rank"] = feats["avg_gain"] / feats["avg_qual"].clip(lower=1)


    finish_col = "positionOrder" if "positionOrder" in df_season.columns else "position_finish"
    feats["avg_finish"] = g[finish_col].mean()

    feats["race_count"] = g["position_change"].count()

    feats = feats.reset_index()
    feats = feats[feats["race_count"] >= min_races]

    return feats

In [ ]:
season_records = []
for year, season_df in df.groupby("year"):
    feats = compute_behavioral_features(season_df, groupby_col="driverId")
    feats["year"] = year
    season_records.append(feats)
 
driver_season_df = pd.concat(season_records, ignore_index=True)
driver_season_df = driver_season_df.dropna().reset_index(drop=True)
 

driver_season_df = pd.merge(
    driver_season_df,
    drivers[["driverId", "forename", "surname"]],
    on="driverId",
    how="left"
)
driver_season_df["forename"] = driver_season_df["forename"].fillna("")
driver_season_df["surname"]  = driver_season_df["surname"].fillna("")
driver_season_df["driver_name"] = (
    driver_season_df["forename"].str[:1] + ". " + driver_season_df["surname"]
).str.strip()
 
print(f"Driver-season dataset: {driver_season_df.shape}")
print(driver_season_df.head())

In [ ]:
feature_cols = [
    "avg_gain",
    "gain_pct",
    "volatility",
    "p10_gain",
    "p90_gain",
    "gain_per_qual_rank",
]
 
X = driver_season_df[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
 

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

driver_season_df["pca1"] = X_pca[:, 0]
driver_season_df["pca2"] = X_pca[:, 1]
 
print("Explained variance ratio:", pca.explained_variance_ratio_)

In [ ]:
loadings = pd.DataFrame(
    pca.components_,
    columns=feature_cols,
    index=["PC1", "PC2"]
).round(3)
print("\nPCA loadings:")
print(loadings)

In [ ]:
for pc in ["PC1", "PC2"]:
    top_features = loadings.loc[pc].abs().nlargest(2).index.tolist()
    dominant_direction = "positive" if loadings.loc[pc, top_features[0]] > 0 else "negative"
    print(f"{pc} is primarily driven by: {top_features} ({dominant_direction} direction)")

In [ ]:
pc1_top = loadings.loc["PC1"].abs().idxmax()
pc2_top = loadings.loc["PC2"].abs().idxmax()
pc1_label = f"PC1 (dominated by {pc1_top})"
pc2_label = f"PC2 (dominated by {pc2_top})"
print(f"\nSuggested axis labels:\n  x: {pc1_label}\n  y: {pc2_label}")

In [ ]:
inertias, sil_scores = [], []
K_range = range(2, 9)
 
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels_k))
 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(K_range, inertias, marker="o")
ax1.set_xlabel("K"); ax1.set_ylabel("Inertia"); ax1.set_title("Elbow")
ax2.plot(K_range, sil_scores, marker="o", color="darkorange")
ax2.set_xlabel("K"); ax2.set_ylabel("Silhouette"); ax2.set_title("Silhouette Score")
plt.tight_layout()
plt.savefig("../outputs/k_selection.png", dpi=150)
plt.show()

In [ ]:
BEST_K = 3
kmeans = KMeans(n_clusters=BEST_K, random_state=42, n_init=20)
driver_season_df["cluster"] = kmeans.fit_predict(X_scaled)

In [ ]:
observed_sil = silhouette_score(X_scaled, driver_season_df["cluster"])
print(f"\nObserved silhouette score: {observed_sil:.4f}")
 
rng = np.random.default_rng(42)
null_scores = []
for _ in range(200):
    shuffled_labels = rng.permutation(driver_season_df["cluster"].values)
    null_scores.append(silhouette_score(X_scaled, shuffled_labels))
 
p_value = np.mean(np.array(null_scores) >= observed_sil)
print(f"Permutation test p-value ({len(null_scores)} permutations): {p_value:.4f}")
if p_value < 0.05:
    print("  → Cluster structure is statistically significant (p < 0.05)")
else:
    print("  → Cluster structure is NOT statistically significant — interpret with caution")
 

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null_scores, bins=30, color="steelblue", edgecolor="white", label="Null distribution")
ax.axvline(observed_sil, color="coral", linewidth=2, label=f"Observed ({observed_sil:.3f})")
ax.set_xlabel("Silhouette score")
ax.set_ylabel("Count")
ax.set_title("Permutation test: observed vs null silhouette scores")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/permutation_test.png", dpi=150)
plt.show()

In [ ]:
df_clean = df[df["completion_pct"] >= 0.90]
 
season_records_clean = []
for year, season_df in df_clean.groupby("year"):
    feats = compute_behavioral_features(season_df, groupby_col="driverId")
    feats["year"] = year
    season_records_clean.append(feats)
 
driver_season_df_clean = pd.concat(season_records_clean, ignore_index=True)
driver_season_df_clean = driver_season_df_clean.dropna().reset_index(drop=True)
 
X_clean = driver_season_df_clean[feature_cols]
X_clean_scaled = scaler.transform(X_clean)
 

In [ ]:
driver_season_df_clean["cluster"] = kmeans.predict(X_clean_scaled)
 
common = pd.merge(
    driver_season_df[["driverId", "year", "cluster"]],
    driver_season_df_clean[["driverId", "year", "cluster"]],
    on=["driverId", "year"],
    suffixes=("_full", "_clean")
)
 
ari = adjusted_rand_score(common["cluster_full"], common["cluster_clean"])
print(f"\nARI (DNF sensitivity): {ari:.4f}")
if ari > 0.7:
    print("  → Clusters are robust to DNF inclusion (ARI > 0.70)")
else:
    print("  → Clusters show sensitivity to DNF inclusion — inspect era breakdown below")
 

In [ ]:
common["agrees"] = common["cluster_full"] == common["cluster_clean"]
cluster_agreement = common.groupby("cluster_full")["agrees"].mean().round(3)
print("\nPer-cluster agreement (full vs DNF-filtered):")
print(cluster_agreement)
 
common_era = pd.merge(common, driver_season_df[["driverId", "year"]], on=["driverId", "year"])
common_era["era"] = pd.cut(
    common_era["year"],
    bins=[0, 1993, 2009, 2030],
    labels=["Pre-1994", "1994–2009", "2010+"]
)
era_agreement = common_era.groupby("era", observed=False)["agrees"].mean().round(3)
print("\nPer-era agreement rate (DNF sensitivity):")
print(era_agreement)
print("  → Higher disagreement in early eras is expected given higher DNF rates pre-1994.")

In [ ]:
cluster_summary = driver_season_df.groupby("cluster")[feature_cols].mean().round(3)
print("\nCluster summary (mean behavioral features):")
print(cluster_summary)
print("\nUse the table above to verify / adjust cluster_names before proceeding.")
print("Expected pattern:")
print("  Consistent Climbers  → high avg_gain, high gain_pct, low volatility")
print("  Volatile Drivers     → high volatility, high p90_gain, lower gain_pct")
print("  Midfield Stabilizers → near-zero avg_gain, moderate volatility")

In [ ]:
cluster_names = {
    0: "Midfield Stabilizers",
    1: "Volatile Drivers",
    2: "Consistent Climbers",
}

driver_season_df["cluster_name"] = driver_season_df["cluster"].map(cluster_names)

In [ ]:
print("\nCluster naming validation:")

for cid, cname in cluster_names.items():
    row = cluster_summary.loc[cid]

    if cname == "Midfield Stabilizers" and row["avg_gain"] < -2:
        print(f"⚠ Cluster {cid}: might not be stabilizers (avg_gain={row['avg_gain']:.2f})")

    elif cname == "Consistent Climbers" and row["avg_gain"] < 1:
        print(f"⚠ Cluster {cid}: weak climbing behavior")

    elif cname == "Volatile Drivers" and row["volatility"] < 5:
        print(f"⚠ Cluster {cid}: low volatility for 'volatile'")

    else:
        print(f"✓ Cluster {cid} ({cname}) looks consistent")

In [ ]:
full_cluster_profile = driver_season_df.groupby("cluster_name")[
    feature_cols + ["avg_qual", "avg_finish"]
].mean().round(2)

print("\nFull cluster profile:")
print(full_cluster_profile)

In [ ]:
plt.figure(figsize=(8, 6))
for cid in sorted(driver_season_df["cluster"].unique()):
    subset = driver_season_df[driver_season_df["cluster"] == cid]
    plt.scatter(subset["pca1"], subset["pca2"],
                label=cluster_names[cid], alpha=0.6, s=20)
 
sample = driver_season_df.sample(12, random_state=42)
for _, row in sample.iterrows():
    plt.text(row["pca1"], row["pca2"], row["driver_name"], fontsize=8)
 
plt.xlabel(pc1_label)
plt.ylabel(pc2_label)
plt.title("Driver behavioral space (PCA projection)")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/pca_behavioral_space.png", dpi=150)
plt.show()

In [ ]:
MIN_SAMPLES_DBSCAN = 5
nn = NearestNeighbors(n_neighbors=MIN_SAMPLES_DBSCAN)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])[::-1]

diffs1 = np.diff(k_distances)
diffs2 = np.diff(diffs1)

n = len(diffs2)
search_start = int(0.20 * n)
search_end   = int(0.80 * n)
elbow_idx = np.argmax(np.abs(diffs2[search_start:search_end])) + search_start + 1
eps_auto  = float(k_distances[elbow_idx])

if eps_auto > 2.0:
    eps_auto = float(np.percentile(k_distances, 10))
    print(f"DBSCAN: heuristic eps was too large — using 10th-percentile fallback: {eps_auto:.4f}")

print(f"\nDBSCAN: auto-selected eps = {eps_auto:.4f} (elbow at index {elbow_idx})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_distances, linewidth=1)
ax.axvline(elbow_idx, color="coral", linestyle="--", label=f"Elbow (eps≈{eps_auto:.2f})")
ax.set_xlabel("Points sorted by distance")
ax.set_ylabel(f"{MIN_SAMPLES_DBSCAN}-NN distance")
ax.set_title("K-distance plot for DBSCAN eps selection")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/dbscan_eps_selection.png", dpi=150)
plt.show()

dbscan = DBSCAN(eps=eps_auto, min_samples=MIN_SAMPLES_DBSCAN)
dbscan_labels = dbscan.fit_predict(X_scaled)
driver_season_df["dbscan_cluster"] = dbscan_labels

outliers_df = driver_season_df[driver_season_df["dbscan_cluster"] == -1]

n_outliers = (dbscan_labels == -1).sum()
n_total    = len(dbscan_labels)
print(f"\nDBSCAN outliers: {n_outliers} / {n_total} ({100*n_outliers/n_total:.1f}%)")
print("DBSCAN cluster distribution:")
print(driver_season_df["dbscan_cluster"].value_counts())



In [ ]:

outliers_df = driver_season_df[driver_season_df["dbscan_cluster"] == -1].copy()

if n_outliers == 0:
    print("\nDBSCAN found 0 outliers with the selected eps.")
    print("This means all driver-seasons are densely connected at this scale.")
    print("Interpretation: the behavioral feature space has no extreme isolates —")
    print("all observed behaviors fall within the core distribution.")
    print("Consider reducing eps manually or increasing min_samples to expose sparse regions.")
    outlier_race_counts     = pd.Series(dtype=float)
    non_outlier_race_counts = driver_season_df["race_count"]
    outlier_feature_means   = pd.Series({f: float("nan") for f in feature_cols})
else:
    outlier_race_counts     = outliers_df["race_count"]
    non_outlier_race_counts = driver_season_df[
        driver_season_df["dbscan_cluster"] != -1
    ]["race_count"]
    outlier_feature_means   = outliers_df[feature_cols].mean().round(3)

print(f"\nOutlier median race_count: {outlier_race_counts.median()}")
print(f"Non-outlier median race_count: {non_outlier_race_counts.median():.1f}")

print("\nOutlier mean feature values:")
print(outlier_feature_means)
print("\nComparison with overall population means:")
print(driver_season_df[feature_cols].mean().round(3))



In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    driver_season_df[driver_season_df["dbscan_cluster"] != -1]["pca1"],
    driver_season_df[driver_season_df["dbscan_cluster"] != -1]["pca2"],
    alpha=0.4, s=15, color="steelblue", label="Core drivers"
)
ax.scatter(
    outliers_df["pca1"], outliers_df["pca2"],
    alpha=0.8, s=40, color="coral", marker="x",
    label=f"DBSCAN outliers (n={n_outliers})"
)
if n_outliers > 0:
    sample_out = outliers_df.sample(min(8, len(outliers_df)), random_state=42)
    for _, row in sample_out.iterrows():
        ax.text(row["pca1"], row["pca2"], row["driver_name"], fontsize=7, color="coral")
ax.set_xlabel(pc1_label); ax.set_ylabel(pc2_label)
ax.set_title("DBSCAN outliers in PCA space")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/dbscan_outliers_pca.png", dpi=150)
plt.show()



In [ ]:

if n_outliers > 0:
    outlier_kmeans_dist = outliers_df["cluster"].map(cluster_names).value_counts()
    print("\nDBSCAN outliers by K-Means cluster (cross-reference):")
    print(outlier_kmeans_dist)
    print("  → Outliers concentrated in a specific K-Means cluster may indicate")
    print("    that cluster boundary is genuinely ambiguous or data-sparse.")
else:
    print("\nDBSCAN outlier cross-reference skipped — no outliers detected.")

In [ ]:
driver_trajectories = (
    driver_season_df
    .sort_values(["driverId", "year"])
    .groupby("driverId")["cluster"]
    .apply(list)
    .reset_index()
    .rename(columns={"cluster": "cluster_sequence"})
)
 
driver_trajectories["n_seasons"]        = driver_trajectories["cluster_sequence"].apply(len)
driver_trajectories["n_unique_clusters"] = driver_trajectories["cluster_sequence"].apply(
    lambda seq: len(set(seq))
)
 
 
def stability_score(seq):
    if len(seq) < 2:
        return None
    return Counter(seq).most_common(1)[0][1] / len(seq)
 
 
driver_trajectories["stability"] = driver_trajectories["cluster_sequence"].apply(stability_score)
 
stable_drivers = driver_trajectories[driver_trajectories["n_seasons"] >= 3].copy()
print(f"\nDrivers with 3+ seasons: {len(stable_drivers)}")
print(stable_drivers["stability"].describe())

transitions = {"from": [], "to": []}
for _, group in driver_season_df.sort_values(["driverId", "year"]).groupby("driverId"):
    seq = group["cluster"].tolist()
    for a, b in zip(seq[:-1], seq[1:]):
        transitions["from"].append(a)
        transitions["to"].append(b)
 
trans_df = pd.DataFrame(transitions)
transition_matrix = pd.crosstab(
    trans_df["from"], trans_df["to"], normalize="index"
).round(3)
 
print("\nTransition matrix (row=from, col=to):")
print(transition_matrix)

In [ ]:
count = 0
print("\nSample notable transitions (driver changed cluster year-over-year):")
for driver_id, group in driver_season_df.sort_values(["driverId", "year"]).groupby("driverId"):
    seq   = group["cluster"].tolist()
    names = group["driver_name"].tolist()
    for i in range(len(seq) - 1):
        if seq[i] != seq[i + 1]:
            print(f"  {names[i]}: {cluster_names[seq[i]]} → {cluster_names[seq[i+1]]}")
            count += 1
            break           
    if count >= 5:
        break        

In [ ]:
print("\nTransition asymmetry analysis:")
asymmetry_records = []
cluster_ids = sorted(transition_matrix.index.tolist())
 
for i in cluster_ids:
    for j in cluster_ids:
        if i >= j:
            continue
        p_ij = transition_matrix.loc[i, j] if j in transition_matrix.columns else 0.0
        p_ji = transition_matrix.loc[j, i] if i in transition_matrix.columns else 0.0
        asymmetry = abs(p_ij - p_ji)
        direction = (
            f"{cluster_names[i]}→{cluster_names[j]}"
            if p_ij >= p_ji
            else f"{cluster_names[j]}→{cluster_names[i]}"
        )
        asymmetry_records.append({
            "pair": f"{cluster_names[i]} / {cluster_names[j]}",
            "P(i→j)": p_ij,
            "P(j→i)": p_ji,
            "asymmetry": round(asymmetry, 3),
            "dominant_direction": direction,
        })
 
asymmetry_df = pd.DataFrame(asymmetry_records).sort_values("asymmetry", ascending=False)
print(asymmetry_df.to_string(index=False))
print("\n  → Large asymmetry values suggest directional career trajectory effects.")
print("    E.g., if P(Consistent Climbers→Volatile) > P(Volatile→Consistent Climbers),")
print("    behavioral decline may be more common than improvement.")

In [ ]:
from scipy.stats import chi2_contingency
count_matrix = pd.crosstab(trans_df["from"], trans_df["to"])
chi2, p_chi2, dof, expected = chi2_contingency(count_matrix.values)
print(f"\nChi-square test of transition matrix vs independence: χ²={chi2:.2f}, p={p_chi2:.4f}")
if p_chi2 < 0.05:
    print("  → Transitions are non-random (p < 0.05) — cluster membership is predictive of future cluster.")
else:
    print("  → No significant structure in transitions detected.")
 
 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = cm.Set2.colors
 
for cid, cname in cluster_names.items():
    sub = driver_season_df[driver_season_df["cluster"] == cid]
    axes[0].scatter(sub["volatility"], sub["gain_pct"],
                    c=[colors[cid]], label=cname, alpha=0.7, s=20)
 
axes[0].set_xlabel("Volatility (std of position change)")
axes[0].set_ylabel("Gain frequency")
axes[0].set_title("Behavioral space")
axes[0].legend()
 
axes[1].hist(stable_drivers["stability"], bins=20, color="steelblue", edgecolor="white")
axes[1].axvline(
    stable_drivers["stability"].median(), color="coral", linestyle="--",
    label=f"Median: {stable_drivers['stability'].median():.2f}"
)
axes[1].set_xlabel("Stability score")
axes[1].set_ylabel("Drivers")
axes[1].set_title("Stability distribution")
axes[1].legend()
 
plt.tight_layout()
plt.savefig("../outputs/behavioral_space.png", dpi=150)
plt.show()

In [ ]:
year_cluster_counts = (
    driver_season_df.groupby(["year", "cluster"]).size().unstack(fill_value=0)
)
ax = year_cluster_counts.plot(kind="bar", stacked=True, figsize=(13, 5),
                               color=[colors[c] for c in sorted(year_cluster_counts.columns)])
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, [cluster_names[int(c)] for c in sorted(year_cluster_counts.columns)],
          title="Cluster")
ax.set_xlabel("Season"); ax.set_ylabel("Driver count")
ax.set_title("Cluster composition across seasons")
plt.tight_layout()
plt.savefig("../outputs/cluster_over_time.png", dpi=150)
plt.show()
 

In [ ]:
name_labels = [cluster_names[c] for c in sorted(transition_matrix.index)]
 
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    transition_matrix,
    annot=True, fmt=".2f", cmap="Blues",
    xticklabels=name_labels,  
    yticklabels=name_labels,
    ax=ax, linewidths=0.4
)
ax.set_title("Season-to-season cluster transition probabilities")
ax.set_xlabel("Next season cluster")
ax.set_ylabel("Current season cluster")
plt.tight_layout()
plt.savefig("../outputs/transition_heatmap.png", dpi=150)
plt.show()
 

In [ ]:
print("\nEra-based analysis:")

year_min = int(driver_season_df["year"].min())
year_max = int(driver_season_df["year"].max())

print(f"Data spans {year_min}–{year_max}")

In [ ]:
if year_min < 1994:
    era_bins = [year_min - 1, 1993, 2009, year_max + 1]
    era_labels = ["Pre-1994", "1994–2009", "2010+"]
else:
    era_bins = [year_min - 1, 2009, year_max + 1]
    era_labels = ["1994–2009", "2010+"]

In [ ]:
driver_season_df["era"] = pd.cut(
    driver_season_df["year"],
    bins=era_bins,
    labels=era_labels
)

In [ ]:
era_cluster_props = (
    driver_season_df
    .groupby(["era", "cluster_name"], observed=True)
    .size()
    .unstack(fill_value=0)
    .apply(lambda row: row / row.sum(), axis=1)
    .round(3)
)

print("\nCluster proportions by era:")
print(era_cluster_props)

In [ ]:
print("\nSilhouette score per era:")

for era_label, era_df in driver_season_df.groupby("era", observed=True):
    if era_df["cluster"].nunique() < 2:
        continue

    era_idx = era_df.index
    X_era = X_scaled[era_idx]
    labels_era = driver_season_df.loc[era_idx, "cluster"].values

    sil_era = silhouette_score(X_era, labels_era)

    print(f"  {era_label}: silhouette = {sil_era:.4f} (n={len(era_df)})")

In [ ]:
era_stability = (
    pd.merge(
        driver_season_df[["driverId", "year", "era"]],
        driver_trajectories[["driverId", "stability"]],
        on="driverId"
    )
    .groupby("era", observed=True)["stability"]
    .mean()
    .round(3)
)

print("\nMean stability score per era:")
print(era_stability)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

era_cluster_props.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[colors[list(cluster_names.values()).index(c)] for c in era_cluster_props.columns]
)

ax.set_xlabel("Era")
ax.set_ylabel("Proportion")
ax.set_title("Cluster composition by regulatory era")

ax.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
score = silhouette_score(X_scaled, driver_season_df["cluster"])
print(f"\nFinal silhouette score (all data): {score:.4f}")

In [ ]:
X_lr = df[["position_qual"]]
y_lr = df["positionOrder"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X_lr, y_lr, test_size=0.2, random_state=42
)
 
lr = LinearRegression().fit(X_train, y_train)
 
results_table = pd.DataFrame({
    "Model": ["Naive baseline (qual = finish)", "Linear Regression"],
    "MAE": [
        mean_absolute_error(y_test, X_test["position_qual"]),
        mean_absolute_error(y_test, lr.predict(X_test))
    ],
    "RMSE": [
        np.sqrt(np.mean((y_test - X_test["position_qual"].values) ** 2)),
        np.sqrt(np.mean((y_test - lr.predict(X_test)) ** 2))
    ]
}).round(3)
 
print("\nSupervised baseline comparison:")
print(results_table)
 

In [ ]:
with zipfile.ZipFile("../data/archive.zip") as z:
    df2   = pd.read_csv(z.open("f1_2024_race_results.csv"))
    qual2 = pd.read_csv(z.open("f1_qualifying_results_2024.csv"))
 
qual2 = qual2.rename(columns={"driver_name": "driver", "qualifying_position": "position_qual"})
df2   = df2.rename(columns={"driver_name": "driver", "position": "position_finish"})
 
df2_merged = pd.merge(df2, qual2, on=["driver", "race_id"], how="inner")
df2_merged["position_change"] = df2_merged["position_qual"] - df2_merged["position_finish"]
df2_merged = df2_merged[
    (df2_merged["position_qual"] > 0) & (df2_merged["position_finish"] > 0)
]

In [ ]:
features_2024 = compute_behavioral_features(
    df2_merged, groupby_col="driver", min_races=3
)
 
X_2024_scaled = scaler.transform(features_2024[feature_cols])
 
features_2024["cluster"]      = kmeans.predict(X_2024_scaled)
features_2024["cluster_name"] = features_2024["cluster"].map(cluster_names)
 
print("\n2024 Driver cluster assignments:")
print(features_2024[["driver", "cluster_name"]].to_string(index=False))


In [ ]:
print("\nBehavioral verification: 2024 cluster means vs historical cluster means")
hist_means  = cluster_summary                                         
modern_means = features_2024.groupby("cluster_name")[feature_cols].mean().round(3)
 
comparison = pd.concat(
    [hist_means.rename(index=cluster_names), modern_means],
    keys=["Historical", "2024"]
)
print(comparison)

In [ ]:
for cname in cluster_names.values():
    if cname not in modern_means.index:
        continue
    hist_row   = hist_means.rename(index=cluster_names).loc[cname]
    modern_row = modern_means.loc[cname]
    mad = (hist_row - modern_row).abs().mean()
    print(f"  {cname}: mean absolute feature deviation (historical vs 2024) = {mad:.3f}")
 
print("\nBaseline MAE (2024, qual position as naive predictor):")
baseline_mae = (df2_merged["position_qual"] - df2_merged["position_finish"]).abs().mean()
print(f"  {baseline_mae:.3f}")
 
print("Average position change (2024):", round(df2_merged["position_change"].mean(), 3))
print("Std of position change (2024):", round(df2_merged["position_change"].std(), 3))

In [ ]:
 
driver_season_df.to_csv("../outputs/driver_season_clusters.csv", index=False)
driver_trajectories.to_csv("../outputs/driver_trajectories.csv", index=False)
transition_matrix.to_csv("../outputs/cluster_transitions.csv")
cluster_summary.to_csv("../outputs/cluster_summary.csv")
features_2024.to_csv("../outputs/driver_clusters_2024.csv", index=False)
asymmetry_df.to_csv("../outputs/transition_asymmetry.csv", index=False)
era_cluster_props.to_csv("../outputs/era_cluster_proportions.csv")
 
print("\nAll outputs saved to ../outputs/")